# Chương 3. Đặc trưng thời gian, dị thường và dự báo

Từ chuỗi đã phân rã ở chương 2: trích xuất đặc trưng thời gian, phát hiện dị thường bằng 3 phương pháp, xây dựng ma trận đặc trưng và huấn luyện Random Forest Regressor dự báo $t+1$ như bằng chứng định lượng cho chất lượng tiền xử lý.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dmprep import config
from dmprep.data import load_main_series, load_sp500, to_wide

config.set_seed()
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

RESULTS: dict = {}  # mọi score / p-value của chương này được lưu vào đây

In [2]:
series = load_main_series()

decomp_path = config.PROCESSED_DIR / "ts_decomposed.parquet"
if decomp_path.exists():
    decomposed = pd.read_parquet(decomp_path)
    print("Đã đọc chuỗi phân rã:", decomposed.shape)
else:
    decomposed = None
    print("⚠ Chưa có ts_decomposed.parquet: hãy hoàn thành mục 2.3 ở chương 2 trước.")

⚠ Chưa có ts_decomposed.parquet: hãy hoàn thành mục 2.3 ở chương 2 trước.


## 3.1 Trích xuất đặc trưng thời gian

- **Mã hóa chu kỳ:** $\sin(2\pi t/T),\ \cos(2\pi t/T)$ cho ngày trong tuần ($T=5$ phiên hoặc 7 ngày) và tháng trong năm ($T=12$). Mã hóa này giúp tháng 12 và tháng 1 "gần nhau".
- **Ngày lễ / sự kiện:** dùng `holidays.financial_holidays("NYSE")`, ví dụ cờ "phiên ngay trước/sau ngày nghỉ".
- **Thống kê cửa sổ trượt:** mean, std, min, max, skewness với nhiều cửa sổ.

⚠ Mọi đặc trưng cửa sổ phải chỉ dùng thông tin **đến thời điểm $t$** (dùng `.shift(1)` khi cần) để tránh rò rỉ dữ liệu tương lai.

In [3]:
# TODO: xây DataFrame `features` index theo ngày
# features = pd.DataFrame(index=series.index)
# dow, month = series.index.dayofweek, series.index.month
# features["dow_sin"], features["dow_cos"] = np.sin(2*np.pi*dow/5), np.cos(2*np.pi*dow/5)
# features["month_sin"], features["month_cos"] = np.sin(2*np.pi*(month-1)/12), np.cos(2*np.pi*(month-1)/12)
# ... cờ ngày lễ ...
# for w in [5, 21, 63]:
#     r = log_ret.rolling(w)
#     features[f"roll_mean_{w}"] = r.mean(); ... std, min, max, skew

**Phân tích kết quả.**

> TODO: trực quan một vài đặc trưng, giải thích ý nghĩa từng nhóm.

## 3.2 Phát hiện dị thường

So sánh 3 phương pháp:

1. **Z-score trên chuỗi đã deseasonalize** (thường dùng log-return): $z_t = \frac{x_t-\mu}{\sigma}$, dị thường khi $|z_t| > 3$.
2. **Isolation Forest trên cửa sổ trượt:** mỗi điểm được biểu diễn bởi vector cửa sổ $[x_{t-w+1},\dots,x_t]$; điểm dễ bị cô lập (đường đi ngắn trong cây) là dị thường.
3. **Dựa trên tái tạo:** ngưỡng trên phần dư STL ($|R_t| > k\cdot\text{MAD}$) hoặc Autoencoder 1D (PyTorch), dị thường khi lỗi tái tạo lớn.

In [4]:
# TODO: cài đặt 3 phương pháp, lưu cờ dị thường vào DataFrame `anomalies` (mỗi cột một phương pháp)
# Báo cáo: số điểm dị thường mỗi phương pháp, Jaccard giữa các tập, và vẽ đánh dấu trên biểu đồ giá.
# from sklearn.ensemble import IsolationForest

**Phân tích kết quả.**

> TODO: đối chiếu các điểm dị thường với sự kiện thực tế (khủng hoảng 2008, COVID 3/2020, công bố kết quả kinh doanh, chia tách cổ phiếu...).

## 3.3 Ma trận đặc trưng và dự báo bằng Random Forest Regressor

Ma trận đặc trưng gồm: lag $y_{t-1},\dots,y_{t-p}$ với $p$ chọn từ PACF (chương 1), rolling statistics nhiều tỉ lệ, đặc trưng thời gian (3.1), các thành phần phân rã (chương 2). Mục tiêu: dự báo $y_{t+1}$.

- Chia train/test **theo thời gian** (không xáo trộn), ví dụ 80% đầu / 20% cuối.
- Nên so sánh với **baseline ngây thơ** $\hat y_{t+1} = y_t$ để kết luận tiền xử lý có thực sự giúp ích không.
- Cân nhắc dự báo log-return thay vì giá (giá không dừng → RF không ngoại suy được ra ngoài khoảng giá đã thấy).

In [5]:
P_LAGS = 5  # TODO: chọn từ PACF ở chương 1

# TODO:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error
# X, y = ..., ...
# split = int(len(X) * 0.8)
# rf = RandomForestRegressor(n_estimators=300, random_state=config.SEED, n_jobs=-1).fit(X[:split], y[:split])
# pred = rf.predict(X[split:])
# RESULTS["rf_mae"] = mean_absolute_error(y[split:], pred)
# RESULTS["rf_rmse"] = np.sqrt(mean_squared_error(y[split:], pred))
# + baseline, + feature importance

**Phân tích kết quả.**

> TODO: so sánh RF với baseline, các đặc trưng quan trọng nhất, ý nghĩa của kết quả đối với chất lượng tiền xử lý.

## 3.4 [Nâng cao] Granger causality

$X$ *Granger-cause* $Y$ nếu các giá trị quá khứ của $X$ giúp dự báo $Y$ tốt hơn so với chỉ dùng quá khứ của $Y$. Kiểm định F so sánh hai mô hình:

$$Y_t = \sum_{i=1}^{p}\alpha_i Y_{t-i} + \varepsilon_t \quad\text{vs}\quad Y_t = \sum_{i=1}^{p}\alpha_i Y_{t-i} + \sum_{i=1}^{p}\beta_i X_{t-i} + \varepsilon_t$$

Yêu cầu chuỗi **dừng** → dùng log-return của các mã trong `config.TS_GRANGER_TICKERS`.

In [6]:
print("Các mã dùng cho Granger:", config.TS_GRANGER_TICKERS)

# TODO:
# from statsmodels.tsa.stattools import grangercausalitytests
# import networkx as nx
# wide = to_wide(load_sp500(tickers=config.TS_GRANGER_TICKERS))
# rets = np.log(wide).diff().dropna()
# Lập ma trận p-value (hàng = nguyên nhân X, cột = kết quả Y), lấy min p-value qua các lag.
# Vẽ đồ thị có hướng với các cạnh X→Y có p < 0.05 (nhớ cân nhắc hiệu chỉnh đa kiểm định).

Các mã dùng cho Granger: ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ']


**Phân tích kết quả.**

> TODO: diễn giải ma trận nhân quả; nhấn mạnh Granger causality ≠ quan hệ nhân quả thật.

## Lưu kết quả

Toàn bộ con số của chương được ghi ra `data/processed/results/` để chương báo cáo tham chiếu.

In [7]:
config.save_results("03_ts_features_forecast", RESULTS)
RESULTS

{}